# 04 — ตรวจว่า OCR ตรงต้นฉบับแค่ไหน

ประเมินข้อความจาก notebook 03 และเปรียบเทียบโมเดลอื่นด้วย **reference หน้าเดียวกัน กติกาเดียวกัน** ไม่รัน OCR ซ้ำ ใช้ CPU ได้

มีแค่ CER, WER อังกฤษ, กราฟ CER รายหน้า 1 รูป และ diff ข้อความ คู่มือ: `04_ocr_evaluation_GUIDE.md`

ยังไม่มี reference ที่คนตรวจครบหน้า จึงยังไม่มีคะแนนจริง ไม่ถือข้อความจาก PDF text layer ว่าถูกจนกว่าจะเทียบภาพ

In [ ]:
# Colab ติดตั้งใน kernel ปัจจุบัน; ในเครื่องใช้ uv sync --dev จาก Backend
try:
    import google.colab
except ImportError:
    print('Local: use the Backend .venv kernel (uv sync --dev).')
else:
    get_ipython().run_line_magic('pip', '-q install jiwer==4.0.0 "matplotlib>=3.9,<4"')


## 1. เลือกภาษาและไฟล์

ใน Colab เปิด mount Drive ด้านล่าง และเตรียม `Data/jsonl/` กับ `Data/ground_truth/en/` ตามโปรเจกต์ ในเครื่องเปิด notebook จากโฟลเดอร์โปรเจกต์

เพิ่มโมเดลใหม่ใน `MODELS` เมื่อมีผลแล้ว แต่ละ JSONL ต้องมี `pdf_page`, `text`, `language`, `source_file` ตามตัวอย่าง Typhoon ชื่อโมเดลในตารางใช้ label ที่เรากำหนด ไม่เรียก API หรือดาวน์โหลดโมเดลใน notebook นี้

In [ ]:
import csv
import json
import re
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

import jiwer
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

# Colab: เอา # ออกแล้วเชื่อม Drive ด้วยตัวเอง
# from google.colab import drive
# drive.mount('/content/drive')

project_root = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / 'Data' / 'jsonl').is_dir()), None)
DATA_ROOT = (project_root / 'Data' if project_root is not None
             else Path('/content/drive/MyDrive/PDPA_RAG/Data'))
LANGUAGE = 'en'  # เปลี่ยนเป็น 'th' เพื่อวัดภาษาไทยแยกกัน
MODELS = {
    'Typhoon': DATA_ROOT / 'jsonl' / f'pdpa_{LANGUAGE}_typhoon_ocr.jsonl',
    # 'Other model': DATA_ROOT / 'jsonl' / 'pdpa_en_other_ocr.jsonl',
}
REFERENCE_DIR = DATA_ROOT / 'ground_truth' / LANGUAGE
SAVE_REPORT = False
REPORT_DIR = (project_root / 'experiments' / 'ocr-evaluation-runs' if project_root is not None
              else DATA_ROOT.parent / 'ocr-evaluation-runs')
print('Data:', DATA_ROOT, '| language:', LANGUAGE)

## 2. เตรียม reference ด้วยคนก่อน

เริ่มอังกฤษ 5 หน้า เช่น **1, 10, 20, 36, 39** รวมหน้าที่ทราบปัญหาเพื่อดู diff แต่คะแนนชุดนี้เป็นชุดวินิจฉัย ไม่ใช่ข้อสรุปทั้งเล่ม

เปิดภาพ PDF แล้วเขียน plain text **ครบหน้า** เป็น `Data/ground_truth/en/page_001.txt` เป็นต้น รวม header และคำบรรยายที่เห็นจริง ไม่ใส่ Markdown หรือเลขหน้าที่เป็น header/footer ใช้ PDF text layer เป็นร่างได้ แต่ตรวจจากภาพทุกบรรทัดก่อนบันทึก

ไฟล์ในโฟลเดอร์นี้ถือว่าเจ้าของตรวจแล้ว: อย่าใส่ร่างที่ยังไม่ตรวจหรือคัดลอก OCR มาให้คะแนนตัวเอง ถ้าร่างยังไม่เสร็จให้เก็บที่อื่น ไทยทำแบบเดียวกันใน `ground_truth/th/`

ไม่มี reference → ไม่คำนวณคะแนน ไม่มีการสร้าง reference อัตโนมัติ

## 3. โหลดข้อมูลและกำหนดกติกาข้อความ

ใช้ `content_v1`: Unicode NFC, ยุบ whitespace, ตัดแท็กเลขหน้าและเครื่องหมายหัวข้อ/emphasis ที่รองรับ เก็บตัวอักษรจริง ตัวเลข คำปฏิเสธ ตัวพิมพ์ และคำบรรยาย figure ไว้ ไม่แก้คำสะกดให้โมเดล

นี่เป็นกติกาแบบง่าย ไม่ใช่ Markdown parser ครบทุกกรณี ถ้า OCR โมเดลใหม่มี wrapper ต่างกัน ให้ปรับกติกากับทั้งสองฝั่งเหมือนกันและบันทึกการเปลี่ยนไว้ ห้ามลบตัวเลขทั่วไปเพื่อเพิ่มคะแนน

In [ ]:
def normalize(text):
    text = unicodedata.normalize('NFC', text)
    text = re.sub(r'<page_number>.*?</page_number>', '', text, flags=re.S)
    text = re.sub(r'</?figure>', '', text)
    text = re.sub(r'^\s{0,3}#{1,6}\s+', '', text, flags=re.M)
    text = re.sub(r'(\*\*|__)(.+?)\1', r'\2', text)
    text = re.sub(r'(?<!\w)\*([^*\n]+)\*(?!\w)', r'\1', text)
    return re.sub(r'\s+', ' ', text).strip()


def load_model(path, language):
    expected_source = {'th': 'PDPA (TH).pdf', 'en': 'PDPA (Eng).pdf'}[language]
    pages = {}
    for line in Path(path).read_text(encoding='utf-8-sig').splitlines():
        record = json.loads(line)
        page = record['pdf_page']
        if type(page) is not int or page < 1 or page in pages:
            raise ValueError(f'Invalid/duplicate PDF page in {path}')
        if record['language'] != language or record['source_file'] != expected_source:
            raise ValueError(f'Wrong language/source in {path}')
        if not isinstance(record['text'], str):
            raise ValueError(f'text must be a string: {path}, page {page}')
        pages[page] = normalize(record['text'])
    return pages


references = {}
for path in sorted(REFERENCE_DIR.glob('page_*.txt')):
    match = re.fullmatch(r'page_(\d+)\.txt', path.name)
    if not match:
        raise ValueError(f'Use page_001.txt naming: {path.name}')
    page = int(match.group(1))
    text = normalize(path.read_text(encoding='utf-8-sig'))
    if page < 1 or page in references or not text:
        raise ValueError(f'Empty reference or invalid/duplicate page: {path.name}')
    references[page] = text

predictions = {name: load_model(path, LANGUAGE) for name, path in MODELS.items()}
for name, pages in predictions.items():
    missing = set(references) - set(pages)
    if missing:
        raise ValueError(f'{name} missing reference pages: {sorted(missing)}')
print('Reference pages:', sorted(references), '| models:', list(predictions))

## 4. คำนวณ CER / WER

CER = (ตัวอักษรที่แทนที่ + ลบ + เพิ่ม) / จำนวนตัวอักษรอ้างอิง ค่ายิ่งต่ำยิ่งดี และอาจเกิน 100% WER เป็นแบบเดียวกันในระดับคำ ใช้เฉพาะอังกฤษด้วย whitespace tokenizer ไทยแสดง CER โดยนับ Unicode code points หลัง NFC

คะแนนรวมคำนวณจาก errors รวม / ความยาว reference รวม **ไม่เฉลี่ยเปอร์เซ็นต์รายหน้า** ไม่แปลง `1-CER` เป็น accuracy ทางความหมาย เพราะคำผิดเดียวอาจเปลี่ยนจำนวนเงิน อย่าตัดสินจากค่าเฉลี่ยอย่างเดียว

In [ ]:
page_rows, summaries = [], []
for name, pages in predictions.items():
    if not references:
        continue
    ref_texts = [references[p] for p in sorted(references)]
    hyp_texts = [pages[p] for p in sorted(references)]
    total = jiwer.process_characters(ref_texts, hyp_texts)
    summary = {'model': name, 'language': LANGUAGE, 'pages': len(references),
               'cer': total.cer,
               'wer': jiwer.wer(ref_texts, hyp_texts) if LANGUAGE == 'en' else None}
    summaries.append(summary)
    for page in sorted(references):
        result = jiwer.process_characters(references[page], pages[page])
        page_rows.append({'model': name, 'pdf_page': page, 'cer': result.cer,
                          'wer': jiwer.wer(references[page], pages[page]) if LANGUAGE == 'en' else None,
                          'substitutions': result.substitutions,
                          'deletions': result.deletions, 'insertions': result.insertions})

if not summaries:
    print('NOT EVALUATED: prepare human-checked reference text first. No accuracy scores yet.')
for row in summaries:
    word_score = f" | WER {row['wer']:.2%}" if row['wer'] is not None else ''
    print(f"{row['model']}: {row['pages']} pages | CER {row['cer']:.2%}" + word_score)

## 5. กราฟเดียว: CER รายหน้า แยกโมเดล

หาหน้าที่ผิดเยอะและดูว่าโมเดลใหม่แก้หน้าปัญหาได้หรือไม่ ทุกโมเดลถูกวัดบนหน้า reference ชุดเดียวกัน กราฟไม่แทนการตรวจจำนวนเงิน เลขมาตรา และคำปฏิเสธจาก PDF

In [ ]:
figure = None
if page_rows:
    figure, ax = plt.subplots(figsize=(10, 4))
    page_numbers = sorted(references)
    width = 0.8 / len(predictions)
    for index, name in enumerate(predictions):
        scores = {r['pdf_page']: r['cer'] for r in page_rows if r['model'] == name}
        positions = [i - 0.4 + width / 2 + index * width for i in range(len(page_numbers))]
        ax.bar(positions, [scores[p] for p in page_numbers], width=width, label=name)
    ax.set_xticks(range(len(page_numbers)), [str(p) for p in page_numbers])
    ax.set(xlabel='PDF page', ylabel='CER (lower is better)',
           title=f'OCR comparison — {LANGUAGE}, {len(page_numbers)} reviewed pages')
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.legend()
    figure.tight_layout()
    plt.show()
else:
    print('Plot skipped: no reviewed references.')

## 6. เปิดดูว่าผิดตรงไหน

ค่าเริ่มต้นเลือกหน้าที่ CER สูงสุด เปลี่ยน `DIFF_MODEL` / `DIFF_PAGE` เพื่อเทียบหน้าที่ต้องการได้ en/36 ต้องใช้ reference ครบหน้า ไม่ใช่เฉพาะย่อหน้าที่พบปัญหา

In [ ]:
if page_rows:
    worst = max(page_rows, key=lambda row: row['cer'])
    DIFF_MODEL, DIFF_PAGE = worst['model'], worst['pdf_page']
    ref, hyp = references[DIFF_PAGE], predictions[DIFF_MODEL][DIFF_PAGE]
    result = (jiwer.process_words(ref, hyp) if LANGUAGE == 'en'
              else jiwer.process_characters(ref, hyp))
    print(DIFF_MODEL, '| PDF page', DIFF_PAGE)
    print(jiwer.visualize_alignment(result))
else:
    print('Diff skipped: no reviewed references.')

## 7. บันทึกเมื่อพร้อม (ไม่บังคับ)

ตั้ง `SAVE_REPORT=True` ในข้อ 1 เพื่อเก็บ CSV คะแนนรวม/รายหน้าและกราฟในโฟลเดอร์รอบใหม่ ไม่แก้ OCR หรือ reference ไม่มี reference จะไม่บันทึกคะแนน

บันทึกชื่อ/รุ่นโมเดล ค่า OCR ที่ใช้ และหน้า reference ในบันทึกการทดลองด้วย เปลี่ยนโมเดลแล้วใช้ reference เดิมและ normalization เดิม ถ้าดูผลหน้าไหนแล้วใช้ปรับวิธี ให้เก็บหน้าใหม่อีกเล็กน้อยทดสอบการตัดสินใจรอบสุดท้าย

In [ ]:
if SAVE_REPORT and page_rows:
    output_dir = REPORT_DIR / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    output_dir.mkdir(parents=True, exist_ok=False)
    for filename, rows in [('summary.csv', summaries), ('by_page.csv', page_rows)]:
        with (output_dir / filename).open('x', encoding='utf-8', newline='') as output:
            writer = csv.DictWriter(output, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)
    figure.savefig(output_dir / 'cer_by_page.png', dpi=150, bbox_inches='tight')
    print('Saved:', output_dir)
else:
    print('No report written (SAVE_REPORT=False or no reference).')

## แหล่งอ้างอิง

- [LlamaIndex: OCR Accuracy Rate](https://www.llamaindex.ai/glossary/ocr-accuracy-rate): วัดระดับตัวอักษร/คำ เทียบกับต้นฉบับ และระวังจุดสำคัญที่ผิดแม้คะแนนรวมดูดี
- [JiWER](https://jitsi.github.io/jiwer/usage/): CER/WER และ diff ด้วย alignment
- [CER metric card](https://huggingface.co/spaces/evaluate-metric/cer/blob/main/README.md): สูตรและกรณี CER เกิน 100%

ไม่ใช้ตัวเลขเปอร์เซ็นต์หรือคำว่า suitable จากบทความเป็นเกณฑ์รับรองเอกสารเรา และยังไม่สรุปว่า Typhoon ภาษาอังกฤษผิดเยอะทั้งเล่มจากหน้า 36 เพียงหน้าเดียว ต้องวัดกับ reference ก่อนตัดสินใจเปลี่ยนโมเดล